# Kapitel 2

## Frågor

### 1. I kapitlet beskrivs en checklista med sju steg. Beskriv de sju stegen översiktligt. I verkligheten, följs dessa steg i en rak progression eller arbetar man generellt sett mer iterativt?

1. Frame the problem: what are we trying to solve, what is the business goal, what type of ML problem is it (regression, classification etc), and how will the result be used.
2. Get the data: find and collect the data, check licenses and access, look at its size and format.
3. Explore the data: EDA, look for patterns, correlations, missing values and outliers.
4. Prepare the data: cleaning, handling missing values, encoding categorical variables, scaling, feature engineering, splitting into train/validation/test.
5. Select and train models: try several models quickly and shortlist the promising ones.
6. Fine-tune: tune hyperparameters (grid search etc), maybe combine models, evaluate properly on validation data.
7. Present, launch and monitor: present the solution, put it into production, and monitor and maintain it over time (for example retrain when performance drops).

In practice the work is iterative rather than a straight line. While exploring the data you may realize you need other data (back to step 2), or while fine-tuning you may find that the features aren't good enough (back to step 4). After launch you keep coming back to earlier steps as new data arrives or performance drifts. So it's more of a loop than a checklist you go through once.

### 2. Vad menas med att en modell produktionssätts?

It means the model is moved from the notebook or experiment into a real system where it makes predictions that are actually used, for example behind an API that receives requests, and its predictions affect real decisions like recommending a product or approving a loan.

This usually also needs monitoring, logging, a way to retrain the model later, and handling of scaling and uptime. That's a lot more than just training a model in a notebook.

### 3. Vad är scikit-learn för något? Biblioteket följer några centrala designprinciper. Vilka är dessa? Vad är estimators, predictors och transformers?

scikit-learn is an open source Python library for machine learning. It has algorithms for regression, classification and clustering, plus tools for preprocessing, model selection and evaluation. It's built on NumPy and SciPy.

Main design principles:
- **Consistency**: all objects share a simple interface (`fit`, `predict`, `transform` etc), so once you know one estimator you can use the others.
- **Inspection**: hyperparameters are available as public attributes (`model.alpha`), and learned parameters have a trailing underscore (`model.coef_`).
- **Nonproliferation of classes**: datasets are regular NumPy arrays, SciPy sparse matrices or pandas DataFrames, not special classes.
- **Composition**: existing building blocks are reused and combined, for example a Pipeline chains transformers and an estimator.
- **Sensible defaults**: most parameters have reasonable default values, so you can get a baseline running quickly.

The three terms:
- **Estimator**: any object that learns parameters from data with `.fit(X, y)` (or `.fit(X)` for unsupervised). Every model and every transformer is an estimator.
- **Predictor**: an estimator that can also make predictions with `.predict(X)`, like LinearRegression. Usually also has `.score()`.
- **Transformer**: an estimator that transforms data with `.transform(X)`, like StandardScaler or OneHotEncoder. Often has `.fit_transform(X)` that does both steps at once.

### 4. Vad är TensorFlow och Keras?

TensorFlow is an open source library from Google for numerical computation, mainly used for deep learning. It handles building neural networks, running computations on GPU/TPU, automatic differentiation for backpropagation, and deploying models (TF Lite for mobile, TF Serving for production etc).

Keras is a high-level API for building neural networks. You stack layers together instead of writing low-level code yourself, which makes it much faster to build and train a network. `tf.keras` is the version integrated in TensorFlow, but Keras 3 is multi-backend and can also run on JAX or PyTorch. So Keras is the easy-to-use interface, and TensorFlow (or another backend) does the actual computation underneath.

### 5. Kalle och Stina diskuterar maskininlärning över en lunch. Kalle säger “om jag tränat en modell och den inte presterar bra nog på testdatan så justerar jag den tills den gör det.” Stina säger “det är ett stort fel att göra så, det enda du då åstadkommer är att du överanpassar testdatan. Hela syftet med testdatan försvinner då”. Vad säger du om deras dialog?

Stina is right. The test data should only be used once, at the end, to get an honest estimate of how the model performs on data it hasn't seen. If Kalle keeps adjusting the model and checking against the test set, he is using it as a validation set, and information from the test data leaks into his decisions. The model ends up tuned to that particular test set, which is overfitting to the test set instead of the training set.

Then the test score is no longer an unbiased estimate. Kalle should use a separate validation set or cross-validation to tune and compare models, and only use the test set once he has chosen the final model.

### 6. Många AI/ML projekt uppnår inte de ursprungligen satta målen eller att ens passera någon form av prototyp-stadie. Vad tror du detta beror på och hur ska vi förhålla oss till det?

Some reasons I think this happens:
- The problem isn't framed well from the start: it's unclear what the model should solve or how success is measured.
- Data problems: too little data, biased data, or data that doesn't match what happens in production.
- The gap between a working prototype and a reliable production system is underestimated (monitoring, scaling, edge cases, data drift).
- No clear link to business value, so even a technically working model isn't used or doesn't get resources for maintenance.
- Poor collaboration between the data scientists and the people who know the domain and the business.

How to deal with it: expect to iterate (connects to question 1) rather than a straight path from idea to production. Start with a clear, measurable goal and a simple baseline before trying complex models. Spend time on understanding and cleaning the data early, since that's often where the real problems are. And be realistic: many prototypes won't reach production, so it's better to test ideas cheaply and early than to spend a lot of resources before knowing if something works.

### 8. Förklara vad koden nedan gör. Varför är det viktigt att kunna spara en modell?

In [1]:
from sklearn.datasets import make_regression
from sklearn.linear_model import LinearRegression
from joblib import dump, load

X, y = make_regression(n_samples=20000, n_features=3, noise=0.1)
model = LinearRegression().fit(X, y)
dump(model, "linear_model.joblib")
model_loaded = load("linear_model.joblib")
print(model_loaded.predict(X[:5]))

[  13.0198336  -169.28763101   72.96464379   -6.24550272  -52.38648197]


What the code does:
1. `make_regression(...)` generates a synthetic regression dataset with 20000 rows and 3 features, with some noise.
2. Trains a LinearRegression model on it with `.fit(X, y)`.
3. `dump(model, "linear_model.joblib")` saves the trained model to a file with joblib, which works well for objects containing large NumPy arrays like sklearn models.
4. `load(...)` loads the saved model back into a new variable.
5. It predicts the first 5 rows with the loaded model, to show it still works after saving and loading.

Why saving a model matters: training can take a long time, so we don't want to retrain every time we need the model. Saving it means we train once and can reuse the exact same model later, in another script or session, or deploy it on a server to make predictions without the training code or data. It's a basic step towards putting a model into production.

### 9. Denna uppgift består av flera steg enligt nedan.

#### a) Läs in datasetet “data_01.csv” med funktionen read_csv() från Pandas.

In [2]:
import pandas as pd
import numpy as np

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import mean_squared_error, mean_absolute_error

data = pd.read_csv("./data/data_01.csv")
data.head()

,x1,x2,x3,x4,x5,target
0,0.743487,1.072825,1.332911,-1.244771,0.344978,220.173943
1,0.835264,0.202184,0.966480,0.745883,-0.033773,175.873929
2,-1.103234,0.030615,-0.140385,0.727683,-2.831224,-162.270054
3,1.210186,1.685258,-0.394123,0.719024,-2.166585,165.930461
4,0.474577,0.647737,-0.451812,-0.409472,-0.051473,43.250511


#### b) Dela upp datasetet i X och y.

In [3]:
X = data.drop(columns="target")
y = data.target

#### c) Dela upp datan ytterligare, i ett träning-, ett validering-, och ett testset med train_test_split(). Låt 20% av datan vara testdata och 15% av den återstående datan vara valideringsdata.

In [4]:
X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.15, random_state=42)

#### d) Träna två valfria regressionsmodeller (exempelvis LinearRegression och DecisionTreeRegressor) på träningsdatan.

In [5]:
linear_reg = LinearRegression()
linear_reg.fit(X_train, y_train)

tree_reg = DecisionTreeRegressor(random_state=42)

hyper_params = {"max_depth": (None, 1, 2, 3, 5, 10)}

reg = GridSearchCV(
    tree_reg,
    hyper_params,
    scoring="neg_root_mean_squared_error",
    cv=5
)
reg.fit(X_train, y_train)
print(reg.best_params_)

{'max_depth': None}


For the decision tree I also ran a small grid search over `max_depth`, since a tree without a depth limit keeps splitting until the leaves are pure and easily overfits.

#### e) Utvärdera modellerna på valideringsdatan.

In [6]:
def regression_metrics(y_true, y_pred):
    return {
        "MAE": mean_absolute_error(y_true, y_pred),
        "RMSE": np.sqrt(mean_squared_error(y_true, y_pred))
    }

linear_reg_pred = linear_reg.predict(X_val)
tree_reg_pred = reg.predict(X_val)

validation_results = pd.DataFrame({
    "Linear Regression": regression_metrics(y_val, linear_reg_pred),
    "Decision Tree": regression_metrics(y_val, tree_reg_pred)
}).T

validation_results

,MAE,RMSE
Linear Regression,2.760558,3.592301
Decision Tree,78.378486,99.479103


In [7]:
tree_train_rmse = np.sqrt(mean_squared_error(y_train, reg.predict(X_train)))
tree_val_rmse = np.sqrt(mean_squared_error(y_val, tree_reg_pred))
print("Decision tree train RMSE:", tree_train_rmse)
print("Decision tree validation RMSE:", tree_val_rmse)

Decision tree train RMSE: 0.0
Decision tree validation RMSE: 99.47910269452692


Linear Regression did much better on the validation data (MAE 2.76, RMSE 3.59) than the Decision Tree (MAE 78.4, RMSE 99.5). The tree's training RMSE is 0 while its validation RMSE is 99.5, so it fits the training data perfectly but doesn't generalize, which is overfitting. The grid search picked `max_depth=None`, but every depth option did badly on this data. I continue with Linear Regression.

#### f) Träna om den bäst presterande modellen på både tränings- och valideringsdatan.

In [8]:
best_model = LinearRegression().fit(X_train_full, y_train_full)

We retrain on train+val because more data usually gives a better model, and we don't need the validation set separately anymore since it has already been used to choose the model.

#### g) Utvärdera modellen på testdatan.

In [9]:
test_pred = best_model.predict(X_test)
test_results = pd.Series(regression_metrics(y_test, test_pred), name="Test set")
test_results

MAE     2.685240
RMSE    3.371549
Name: Test set, dtype: float64

#### h) Träna om modellen på hela datasetet.

In [10]:
final_model = LinearRegression().fit(X, y)

When we're done evaluating, we retrain on all data (train+val+test) to get the best possible final model. We don't evaluate this model again, the performance estimate is the one from the test set in step g.

### 10. Datasetet “salary_dataset.csv” innehåller 29 observationer av personer. YearsExperience är hur många år de arbetat, och Salary är deras lön

#### a) Läs in datasetet “salary_dataset.csv” med pandas read_csv()-funktion och dela upp i X och y. Avgör själv vilken variabel som ska vara den beroende variabeln y.

In [11]:
from sklearn.model_selection import train_test_split, cross_validate
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor

salary_df = pd.read_csv("./data/salary_dataset.csv")
salary_df.head()

X = salary_df[["YearsExperience"]]
y = salary_df["Salary"]

Salary is what we want to predict, so it's y. YearsExperience is used to predict it, so it's X.

#### b) Dela upp datasetet i träning- och testset. (Inget validerings-set alltså!)

In [12]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

The assignment says 29 observations, but the CSV I got has 30 rows. With test_size=0.2 that gives 24 training rows and 6 test rows. Since the dataset is so small I use cross-validation on the training set instead of also taking out a separate validation set.

#### c) Träna två regressionsmodeller med k-delad korsvalidering med cross_validate()-funktionen från scikit-learn. Använd neg_root_mean_squared_error som scoring. Välj själv hur många iterationer den ska göra genom hyperparametern cv.

In [13]:
linear_reg = LinearRegression()
tree_reg = DecisionTreeRegressor(random_state=42)

linear_scores = cross_validate(linear_reg, X_train, y_train, scoring="neg_root_mean_squared_error", cv=5)
tree_scores = cross_validate(tree_reg, X_train, y_train, scoring="neg_root_mean_squared_error", cv=5)

print("Linear Regression RMSE:", -linear_scores["test_score"].mean())
print("Decision Tree RMSE:", -tree_scores["test_score"].mean())

Linear Regression RMSE: 5293.203196998776
Decision Tree RMSE: 5611.57858424216


In [14]:
linear_train = cross_validate(linear_reg, X_train, y_train, scoring="neg_root_mean_squared_error", cv=5, return_train_score=True)
tree_train = cross_validate(tree_reg, X_train, y_train, scoring="neg_root_mean_squared_error", cv=5, return_train_score=True)

print("Linear Regression train RMSE:", -linear_train["train_score"].mean())
print("Decision Tree train RMSE:", -tree_train["train_score"].mean())

Linear Regression train RMSE: 5156.174831779521
Decision Tree train RMSE: 112.24269429778899


I used cv=5. With 24 training rows that gives folds of about 5 rows, cv=10 would leave only 2-3 rows per validation fold.

#### d) Utvärdera modellen som presterar bäst på testsetet.

In [15]:
# swap depending on which one had lower RMSE above
best_model = LinearRegression().fit(X_train, y_train)

from sklearn.metrics import mean_squared_error
test_pred = best_model.predict(X_test)
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, test_pred)))

Test RMSE: 7059.043621901506


Linear Regression had the lower cross-validation RMSE (5293 vs 5611 for the tree). The tree's training RMSE is about 112 while its CV RMSE is about 5612, a big gap that points to overfitting, while linear regression is about 5156 on training and 5293 in CV. With only one feature and a roughly straight-line relationship, linear regression fits well.

The test RMSE (7059) is higher than the CV estimate, but the test set is only 6 rows, so a couple of points can move it a lot.

### 11. I denna uppgift kommer vi arbeta med kategorisk data.

#### a) Läs in datasetet mpg med seaborns load_dataset()-funktion (det gjordes i koden ovan).

In [16]:
import seaborn as sns

df = sns.load_dataset("mpg")
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin,name
0,18.0,8,307.0,130.0,3504,12.0,70,usa,chevrolet chevelle malibu
1,15.0,8,350.0,165.0,3693,11.5,70,usa,buick skylark 320
2,18.0,8,318.0,150.0,3436,11.0,70,usa,plymouth satellite
3,16.0,8,304.0,150.0,3433,12.0,70,usa,amc rebel sst
4,17.0,8,302.0,140.0,3449,10.5,70,usa,ford torino


#### b) Droppa rader med saknade värden med dropna()-metoden.

In [17]:
df = df.dropna()

#### c) Droppa kolumnen name.

In [18]:
df = df.drop(columns="name")

The name could actually contain useful information, since it includes the make and model, which relate to the kind of car. But with 305 unique values, one-hot encoding would add 305 columns for about 400 rows, so dropping it is reasonable for this exercise. A better option would be extracting just the brand from the name.

#### d) Utför en dummy-variable-encoding på origin-kolumnen med pandas get_dummies()-funktion. Ange drop_first=True så att vi får 2 nya kolumner istället för 3.

In [19]:
df = pd.get_dummies(df, columns=["origin"], drop_first=True)
df.head()

,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin_japan,origin_usa
0,18.0,8,307.0,130.0,3504,12.0,70,False,True
1,15.0,8,350.0,165.0,3693,11.5,70,False,True
2,18.0,8,318.0,150.0,3436,11.0,70,False,True
3,16.0,8,304.0,150.0,3433,12.0,70,False,True
4,17.0,8,302.0,140.0,3449,10.5,70,False,True


`drop_first=True` drops e.g. origin_europe, so we end up with just origin_japan and origin_usa. Europe becomes the reference category, if both new columns are 0 that means the car is from Europe. Avoids the dummy variable trap (perfect multicollinearity between the 3 columns).

#### e) Dela upp datasetet i X och y, med mpg som den beroende variabeln y.

In [20]:
X = df.drop(columns="mpg")
y = df["mpg"]

#### f) Dela upp datasetet i träning- och testset.

In [21]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

#### g) Träna en linjär regressionsmodell på träningsdatan och utvärdera den på testdatan.

In [22]:
lin_reg = LinearRegression().fit(X_train, y_train)

pred = lin_reg.predict(X_test)
print("Test RMSE:", np.sqrt(mean_squared_error(y_test, pred)))
print("Test MAE:", mean_absolute_error(y_test, pred))

Test RMSE: 3.256114096847398
Test MAE: 2.4619996980661436


The mean absolute error on the test set is about 2.46 mpg. RMSE is 3.26, higher than MAE because it gives more weight to the larger errors.

### 12. I avsnitt 2.2 “Ett kodexempel från början till slut - Huspriser i Kalifornien” så gås ett komplett kodexempel igenom. På valideringsdatan fick vi RMSE Random Forest Regression: 52277.96578719621. Försök få ett bättre resultat genom att exempelvis justera hyperparametrar eller genomföra variabelselektion.

In [23]:
from sklearn.datasets import fetch_california_housing
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import GridSearchCV, train_test_split
from sklearn.metrics import mean_squared_error
import numpy as np

housing = fetch_california_housing(as_frame=True)
X = housing.data
y = housing.target

X_train_full, X_test, y_train_full, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_train_full, y_train_full, test_size=0.2, random_state=42)

# baseline, default hyperparameters
rf_baseline = RandomForestRegressor(random_state=42)
rf_baseline.fit(X_train, y_train)
baseline_pred = rf_baseline.predict(X_val)
print("Baseline RMSE:", np.sqrt(mean_squared_error(y_val, baseline_pred)) * 100000)

Baseline RMSE: 52759.737939545645


(RMSE gets multiplied by 100000 above since the California housing target is in units of $100,000s, just so it matches the same scale as the 52277 number from the book.)

In [24]:
hyper_params = {
    "n_estimators": [100, 200, 300],
    "max_depth": [None, 10, 20, 30],
    "max_features": ["sqrt", "log2"],
}

grid = GridSearchCV(
    RandomForestRegressor(random_state=42),
    hyper_params,
    scoring="neg_root_mean_squared_error",
    cv=3,
    n_jobs=-1,
)
grid.fit(X_train, y_train)

print("Best params:", grid.best_params_)

tuned_pred = grid.best_estimator_.predict(X_val)
print("Tuned RMSE:", np.sqrt(mean_squared_error(y_val, tuned_pred)) * 100000)

Best params: {'max_depth': 20, 'max_features': 'log2', 'n_estimators': 300}


Tuned RMSE: 50600.2832026551


Tuning reduced validation RMSE from 52759.74 to 50600.28 on my split, about 4.1%. The selected settings were 300 trees, max_depth=20 and max_features='log2'. I changed several settings together, so this doesn't show which one mattered most. My number is lower than the book's 52277.97, but I can't compare them directly: I used `fetch_california_housing` from sklearn, which doesn't have exactly the same columns and preprocessing as the book's housing.csv pipeline, and the split is different.

I didn't do variable selection. Random forests usually handle weak features reasonably well since those just get picked for fewer splits, but checking `feature_importances_` and dropping the weakest features would be the next thing to try.